<a href="https://colab.research.google.com/github/dineshigdd/Foothill-NLP/blob/main/NLP_cs_12b_module_2_practice_activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Module 2 Practice Activity

**You do not need to submit this Notebook.** It is for practice only.

To begin, select "Copy to Drive" at the top (or alternatively download the Notebook file to a local machine for use with Jupyter).

This instance of the Notebook belongs to you, and you are encouraged to make changes to it. Experimentation often leads to new insights and deeper understanding.

If you ever need to revert back to the original version, you can simply make a new copy by accessing it from Canvas again.

For brevity, descriptions are not provided for the setup steps below. See the **Module 1 Practice Activity** Notebook for explanations of these steps.

### Goals
- Learn about the training and evaluation steps of a feedforward network
- Add optimization techniques to your model with Weight Initialization, Batch Normalization, and Dropout
- Practice building a regression model for the California Housing dataset

### Learn by Experimenting
Please do not simply click "Run all" at the top before reading, as much of the value of this document is to serve as an interactive exploration from one section to the next. In fact, it will help significantly with learning if you first try to understand what the code in a cell might be doing before you run it and get the answer.

Additionally, as you read through the Notebook, you are highly encouraged to change the values in cells to see how the output changes when the cells are re-run. In doing so, you will gain a better understanding of how the various operations behave.

### Notebook Troubleshooting
If you get an unexpected error, the fix is often to click "Run before" from the Runtime menu to run all cells prior to the currently selected cell. If the error persists and you are not sure how to continue, please post on our Canvas forum or send me a message directly with information about the problem.

---

## Notebook Setup

### Import Modules

In [1]:
import numpy as np

from sklearn import datasets

import torch
import torch.nn as nn
from torch.utils import data

### Set a Random Seed Value

In [2]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

### Select a GPU Device

If none available, defaults to the CPU.

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

## Prepare Data

### Load the MNIST Dataset

As before, MNIST is loaded to demonstrate the model. However, here the features are not scaled since Batch Normalization is used (see below).

In [10]:
bunch_obj = datasets.fetch_openml("mnist_784", as_frame=False)
X, y = bunch_obj.data, bunch_obj.target
X.shape, y.shape

((70000, 784), (70000,))

### Convert Labels to Integers

In [11]:
y = y.astype(np.int64)

### Split Training and Test Samples

In [12]:
X_train, X_test, y_train, y_test = X[:10000], X[-1000:], y[:10000], y[-1000:]
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((10000, 784), (1000, 784), (10000,), (1000,))

### Create the Data Loaders

In [13]:
def create_data_loader(X, y, batch_size=32):
    return data.DataLoader(
        data.TensorDataset(
            torch.FloatTensor(X),
            torch.LongTensor(y)
        ),
        batch_size=batch_size,
        shuffle=True
    )

In [14]:
train_loader = create_data_loader(X_train, y_train)
test_loader = create_data_loader(X_test, y_test)

## Define the Model Architecture

In the previous practice activity, a simple feedforward model was used without any regularization. Here, additional techniques are defined and used to help stabilize training and manage the bias-variance tradeoff.

In [ ]:
n_features = X_train.shape[1]
n_features

In [ ]:
n_classes = len(np.unique(y_train))
n_classes

### Weight Initialization

In a fully connected network, having all weight parameters initialized to the same value (such as zero) would cause a lot of redundancy; since each neuron in the same layer is receiving the same inputs, then each neuron in the same layer would output the same value - which would be no different than having a layer of one neuron.

This redundancy can be avoided by initializing the weights to random values, so that during training they get updated by different amounts (as their gradients would be different).

Another issue with deep networks is that their gradients - particularly for layers closer to the input layer - can become either very small or very large during backpropagation, known as the vanishing or exploding gradient problem, respectively.

Much research has been conducted on how to properly sample random initial weights to avoid this problem, and the method chosen depends on in part on the activation function for a given layer. The parameter initialization function below uses Kaiming Initialization (also known as He Initialization), which is one of the most common weight initialization strategies for ReLU activation functions.

In [ ]:
def init_params(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)

Because this function will be applied to the entire network, it must first check that the input is a `nn.Linear` object (and not, say, a `nn.ReLU` or `nn.Dropout` object) before initializing the object's parameters.

Why are the bias parameters simply initialized to zeros? Because they are additive and not multiplicative (like the weights), the biases themselves do not cause vanishing or exploding gradients.

#### Optional: What Is the `isinstance` Function?

The Python `isinstance` function returns True when the first argument is of the data type specified by the second argument.

In [ ]:
xs = [1, 2, 3]
isinstance(xs, list)

If the first argument is not of this data type, the function returns False.

In [ ]:
isinstance(xs, tuple)

The `isinstance` function is very useful inside function definitions to perform certain operations only when the function's arguments are of a specific data type.

### Batch Normalization

In addition to weight initialization, parameters can be stabilized further during training by normalizing each neuron's weighted sum of features using the mean and variance of the entire mini-batch for that feature. By standardizing the outputs of each hidden layer, taking this step results in less extreme gradients during backpropagation, which mitigates the vanishing/exploding gradient problem.

Additionally, PyTorch continuously updates a running mean and variance for each neuron across all mini-batches, so that during evaluation these aggregated statistics can be used to normalize validation samples.

The `BatchNorm1d` component performs this normalization step, and should generally be placed between the `nn.Linear` and activation function components.

```python
model = nn.Sequential(
    ...
    nn.Linear(100, 50),
    nn.BatchNorm1d(50),
    nn.ReLU(),
    ...
)
```

Note that with batch normalization, features often do not need to be normalized (as is commonly done for MNIST images by dividing the feature matrix by 255) before feeding them into a network.

### Dropout

As with traditional models like Random Forests, adding randomness to models can reduce its variance. One way to add randomness to networks is with a technique called Dropout, in which inputs are sometimes ignored by the next layer. This process forces the network to not rely on a small subset of inputs to minimize loss.

The main hyperparameter for Dropout is its rate - the probability that a given input will be ignored for one training sample. In the example below, this rate is set to `0.2`, meaning that each input in the preceding layer has a 20% chance of not being used by the next layer.

The Dropout component can be after a layer's activation function (or, in the case of the input layer, after any normalization).

```python
model = nn.Sequential(
    ...
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    ...
)
```

### Instantiate the Model

With these additional regularization pieces, the final model architecture for this task is constructed below (with a blank space used to visually separate layers).

In [ ]:
model = nn.Sequential(
    nn.BatchNorm1d(n_features),
    nn.Dropout(p=0.2),

    nn.Linear(n_features, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.2),

    nn.Linear(128, n_classes)
)

model

Next, apply the [previously-defined](#scrollTo=RLYxywxj0gD8) `init_params` function to the network, which initializes the parameters of any network component of `nn.Linear` type.

In [ ]:
model.apply(init_params)

The model architecture is now complete! The next step is setting up the hyperparameters.

---

## Setting Hyperparameters

### Number of Epochs

An epoch represents one iteration of training in which every sample has been used at least once. Too many epochs will slow down training and can result in overfitting, while too few can cause underfitting.

In [ ]:
n_epochs = 100

### Learning Rate

As discussed with Stochastic Gradient Descent, the learning rate is a hyperparameter that helps control how much the model parameters (its weights and biases) will change with each update.

Unless an adaptive optimization strategy is selected, it is generally best to use a learning rate closer to zero; while doing so slows training, it generally results in more consistent parameter updates and convergence.

In [ ]:
learning_rate = 0.01

### Optimizer

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

### Cost Function

For classification tasks like MNIST, the cross entropy loss is often used. This loss function combines a Softmax activation (which converts the raw output values to probabilities) with a negative log-likelihood loss (which calculates how far off those probabilities are from the true class labels).

In [ ]:
criterion = nn.CrossEntropyLoss()


Note that the type returned by `nn.CrossEntropyLoss()` is itself a callable - in other words, a function object. For this reason, the variable `criterion` can be called during training just like any function:

```python
criterion(y_pred, y_batch)
```

#### Optional: What Is a Function Object?

In many programming languages (such as Python), functions are themselves values that can be stored in variables and even passed as arguments to other functions.

Let's define a function `add_one` that simply increments its input by one and returns it.

In [ ]:
def add_one(x: int) -> int:
    return x + 1

add_one(1)

Now let's store that function in another variable called `increment`. Notice that the use of `add_one` here does not have any parentheses.

In [ ]:
increment = add_one

The function object is now stored in the variable `increment`, which can be used to call the function with parentheses.

In [ ]:
increment(1)

Using functions in this way allows for some very convenient programming paradigms, such as [callbacks](https://en.wikipedia.org/wiki/Callback_(computer_programming)) - however, these paradigms are outside the scope of this course.

---

## Training

To train the model, it needs to first be sent to the GPU. Following that step, training mode must be activated.

For review, the code used to train the model in the previous practice activity is shown below. (This code exists here as text and is not executable.)

```python
model.to(device)
model.train()
for epoch in range(n_epochs):
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
```

The following subsections describe each of the training steps above.

### Move the Model to the Device

Assuming a GPU has been selected, the model can be moved to that device using its `to` method. This step ensures that the model's parameters reside in the GPU's memory.

During training, the data batches will also be moved to the GPU so that the matrix computations can occur much more quickly on the same device.

In [ ]:
model.to(device)

### Set the Model to Training Mode

Given a sample (or batch of samples), the model exhibits different behavior on its inputs depending whether the model is in training or evaluation mode. Thus, it is important to set the appropriate mode for each step.

In [ ]:
model.train()

### Processing a Single Batch

The Python `next` function is only used here to retrieve a single batch from iterating over `train_loader` once. In practice, you can simply loop over this variable (as seen previously and reproduced below).

```python
for X_batch, y_batch in train_loader:
```

In [ ]:
X_batch, y_batch = next(iter(train_loader))
X_batch.shape, y_batch.shape

#### Move the Batches to the Device

Like the model, the batches need to be moved to the GPU to take advantage of its faster matrix processing.

Why not move the entire dataset to the GPU beforehand instead of one batch at a time? Although that would result in reduced transfer time, it is often not feasible to store an entire dataset on the GPU due to its size.

In [ ]:
X_batch, y_batch = X_batch.to(device), y_batch.to(device)
X_batch.device, y_batch.device

#### Make Predictions

With both the model and the batches on the GPU, the model can be called as a function to produce predictions on the feature matrix.

Recall that batches are each composed of 32 samples, and each sample can be classified as one of ten digits. Thus, this step will produce a matrix of size 32x10.

In [ ]:
y_pred = model(X_batch)
y_pred.shape

#### Calculate the Loss/Cost

As described previously, the loss function (`criterion`) is itself a callable, which accepts predictions and targets as its arguments.

Note: since the error is computed across the entire mini-batch, the result is technically known as the cost. However, the terms are often used interchangably, and the variable `loss` is used here to match code in [the optional book](https://learning.oreilly.com/library/view/hands-on-machine-learning/9798341607972/) for this course.

In [ ]:
loss = criterion(y_pred, y_batch)
loss

The log-loss returned by the loss function is a single-element tensor, which can be retrieved using the tensor's `item` method.

In [ ]:
loss.item()

#### Backpropagation

To update the parameters, the algorithm must propagate the error gradient backward - one layer at a time - from the output layer (where the cost is initially calculated) toward the input layer.

However, PyTorch abstracts most of these details, and the gradients can be calculated with a single call of `loss.backward()`.

In [ ]:
loss.backward()

Next, the `step` method of the optimizer performs the actual updates to all the weight and bias parameters, using the gradients from `loss.backward()` and the learning rate provided to the optimizer previously.

In [ ]:
optimizer.step()

By default, PyTorch accumulates gradients (rather than replacing them). Thus, it is critical to zero-out the gradients after using them to update the parameters and before the next gradient calculation.

In [ ]:
optimizer.zero_grad()

## Evaluation

The evaluation section has several key differences from the training section.

As before, this section is reproduced here as text for review.

```python
model.eval()
n_correct = 0
n_samples = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        predicted_labels = torch.argmax(y_pred, dim=1)
        n_correct += (predicted_labels == y_batch).sum().item()
        n_samples += y_batch.size(0)
```

### Set the Model to Evaluation Mode

Just as the model exhibits specific behavior in training mode, its behavior changes in evaluation mode.

In [ ]:
model.eval()

### Disabling Gradient Computation

In the background, PyTorch does a lot of value tracking (building a structure known as a computation graph used for backpropagation), which is not necessary during evaluation.

Using the `torch.no_grad()` context manager, PyTorch will have less to track and will thus be able to run faster. (Additionally, it prevents any accidental updates to the parameters caused by developer error.)

```python
with torch.no_grad():
```

#### Optional: What Is a Context Manager?

In Python, a context manager is a code block in which a certain configuration is active only within that block.

You may have seen a context manager used for opening files, as in the following code that opens a (fake) file called `file.txt` and stores the list of its lines in a variable called `lines`.

```python
with open("file.txt", "r") as fp:
    lines = fp.readlines()
```

However, after this indented block, the file object `fp` is no longer accessible (the file was closed), so its "context" is no longer active.

### Make Predictions

As seen in the training phase, the model predictions - that is, the raw output values (logits) from the output layer - can be retrieved from calling the model as a function on a (mini-batch) feature matrix.

In [ ]:
y_pred = model(X_batch)
y_pred.shape

To simplify this explanation, let's look at the prediction for the first sample in the mini-batch.

In [ ]:
y_pred[0]

These numbers are the raw values from the output layer, where a higher value indicates the network is more confident about that class.

The highest number can be retrieved using the `argmax` function, which returns the **index** of the highest value. (For MNIST, this index also happens to be the digit being predicted, but for other classification problems the index can be mapped to a class name.)

Below, the predictions for the entire mini-batch are shown.

In [ ]:
predicted_labels = torch.argmax(y_pred, dim=1)
predicted_labels

The `dim=1` argument means that the highest value should be determined row-wise (within each inner list), as each inner list represents the prediction of one sample passed through the network.

### Updating the Count Variables

The number of correct predictions per batch can be found with the following line of code, which is broken down and explained below.

```python
(predicted_labels == y_batch).sum().item()
```

Just like with NumPy, PyTorch tensors can be compared using the `==` operator, which results in a tensor of Boolean values of the same size.

For any index where the values are equal, the output has `True` at that index - otherwise, it has `False`.

In [ ]:
bool_vector = (predicted_labels == y_batch)
bool_vector

The Boolean tensor can be added up by treating each `True` value as a one and each `False` value as a zero (which is a very common way to process Booleans numerically).

In [ ]:
n_correct_tensor = bool_vector.sum()
n_correct_tensor

Finally, the resulting sum can be retrieved from the single-element tensor using the `item` method.

In [ ]:
n_correct_tensor.item()

---

## Putting It All Together

Now that the details have been explained, it may be helpful to have all the code in one place for review. Although there are no new techniques in this section, some of the code lines have been simplified.

### Device

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

### Helper Functions

In [ ]:
def create_data_loader(X, y, batch_size=32):
    return data.DataLoader(
        data.TensorDataset(
            torch.FloatTensor(X),
            torch.LongTensor(y)
        ),
        batch_size=batch_size,
        shuffle=True
    )


def init_params(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)

### Dataset

In [ ]:
bunch_obj = datasets.fetch_openml("mnist_784", as_frame=False)
X, y = bunch_obj.data, bunch_obj.target.astype(np.int64)

train_loader = create_data_loader(X[:10000], y[:10000])
test_loader = create_data_loader(X[-1000:], y[-1000:])

### Model

In [ ]:
model = nn.Sequential(
    nn.BatchNorm1d(X.shape[1]),
    nn.Dropout(p=0.2),

    nn.Linear(X.shape[1], 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.2),

    nn.Linear(128, len(np.unique(y)))
).apply(init_params)

### Hyperparameters

In [ ]:
n_epochs = 100
learning_rate = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

### Training

In [ ]:
model.to(device)
model.train()
for epoch in range(n_epochs):
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print("Mean Loss:", total_loss / len(train_loader))

### Evaluation

In [ ]:
model.eval()
n_correct = 0
n_samples = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        predicted_labels = torch.argmax(y_pred, dim=1)
        n_correct += (predicted_labels == y_batch).sum().item()
        n_samples += y_batch.size(0)
n_correct / n_samples

---

## Regression Tasks

Using a feedforward network for regression is no more difficult than for classification. The necessary changes include:
- Use `nn.MSELoss()` as the `criterion` (cost function).
- The final `nn.Linear` component of the model should have an output size of `1` (unless the task is multi-target regression).

### Try It Yourself

Using the California Housing dataset, see if you can construct a feedforward network that predicts median house values.

Recommended: disconnect and delete this runtime (from the Runtime menu in Colab) to ensure you are not relying on any existing variable values from above.

#### Starter Code

In [ ]:
import numpy as np

from sklearn import datasets
from sklearn import model_selection

import torch
import torch.nn as nn
from torch.utils import data

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

The `DataLoader` helper function is recreated here to transform the target vector to a real-numbered `FloatTensor` (rather than an integer `LongTensor`, as was used for MNIST). It also uses `unsqueeze(1)` to reshape the target vector from 1D to 2D, ensuring its dimensions correctly match the model's output predictions.

In [ ]:
from sklearn import model_selection

def create_regression_data_loader(X, y, batch_size=32):
    return data.DataLoader(
        data.TensorDataset(
            torch.FloatTensor(X),
            torch.FloatTensor(y).unsqueeze(1)
        ),
        batch_size=batch_size,
        shuffle=True
    )

bunch_obj = datasets.fetch_california_housing(as_frame=False)
X, y = bunch_obj.data, bunch_obj.target
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2)

train_loader = create_regression_data_loader(X_train, y_train)
test_loader = create_regression_data_loader(X_test, y_test)

#### Model

#### Hyperparameters

#### Training

#### Evaluation

## Optional: The Keras Library

Keras is a popular high-level library that provides additional abstractions for building networks, which means you can accomplish many tasks with less code (at the loss of some configurability).

Below is a simple implementation of a similar network as seen previously in this Notebook. Most of the components should look similar to those seen with PyTorch, so they will not be re-explained here.

For your projects in this course, you are welcome to use either PyTorch or Keras.

### Import Modules

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers

### Load the Dataset

In [ ]:
bunch_obj = datasets.fetch_openml("mnist_784", as_frame=False)
X, y = bunch_obj.data, bunch_obj.target.astype(np.int64)

X_train, y_train = X[:10000], y[:10000]
X_test, y_test = X[-1000:], y[-1000:]

### Build the Model

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(128, kernel_initializer="he_uniform"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.2),

    layers.Dense(len(np.unique(y)))
])


### Set the Hyperparameters

In [ ]:
n_epochs = 100
learning_rate = 0.01
batch_size = 32

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

### Train the Model

In [ ]:
model.fit(
    X_train,
    y_train,
    batch_size=batch_size,
    epochs=n_epochs,
    verbose=0
)

### Evaluate the Model

In [ ]:
_, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

accuracy

---

## Next Steps

Now that you have seen what each step of the training and evaluation process do, as well as some additional steps like batch normalization and dropout, you will be applying these steps to a dataset of your choice.

See Canvas for details about this project.